In [0]:
CATALOG = "healthcare_life_sciences"
SCHEMA  = "qsar"
VOLUME   = "mlflow_traces"
APP_NAME = "aichemy"
EXP_NAME = "/Workspace/Users/hls-dais2026@databricks-events.com/aichemy-vol"   # new experiment (artifact_location is immutable → must be new)

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.ml import ExperimentAccessControlRequest, ExperimentPermissionLevel

w = WorkspaceClient()
APP_SP = w.apps.get(name=APP_NAME).service_principal_client_id
ART = f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{APP_NAME}"
APP_SP, ART

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{APP_SP}`")
spark.sql(f"GRANT USE SCHEMA  ON SCHEMA  {CATALOG}.{SCHEMA} TO `{APP_SP}`")
spark.sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {CATALOG}.{SCHEMA}.{VOLUME} TO `{APP_SP}`")

In [0]:
exp = mlflow.get_experiment_by_name(EXP_NAME)
exp_id = exp.experiment_id if exp else mlflow.create_experiment(EXP_NAME, artifact_location=ART)
w.experiments.set_permissions(experiment_id=exp_id, access_control_list=[ExperimentAccessControlRequest(service_principal_name=APP_SP, permission_level=ExperimentPermissionLevel.CAN_EDIT)])

print("EXPERIMENT_ID =", exp_id, "| artifact_location =", mlflow.get_experiment(exp_id).artifact_location)